# Selección de un modelo preentrenado de Hugging Face para clasificar el sentimiento de reseñas de clientes en español

**Actividad 4 — Gestión de proyectos de inteligencia artificial (Universidad Tecmilenio)**

| | |
|---|---|
| **Objetivo** | Implementar, evaluar y comparar tres modelos preentrenados de Hugging Face que clasifican reseñas en español como *negativo*, *neutral* o *positivo*, y seleccionar el más adecuado considerando calidad predictiva, latencia y recursos. |
| **Caso de uso** | Triaje automático de opiniones de clientes de comercio electrónico: detectar reseñas negativas para escalarlas a servicio al cliente y monitorear la satisfacción por periodo. |
| **Rama de IA** | Procesamiento de lenguaje natural (clasificación de texto). |
| **Dataset** | `SetFit/amazon_reviews_multi_es` (réplica de la partición en español del *Multilingual Amazon Reviews Corpus*), partición `test`: 5 000 reseñas. |
| **Modelos** | 1) `nlptown/bert-base-multilingual-uncased-sentiment` · 2) `cardiffnlp/twitter-xlm-roberta-base-sentiment` · 3) `lxyuan/distilbert-base-multilingual-cased-sentiments-student` |
| **Tipo de uso** | Solo inferencia con pesos preentrenados (no se realiza *fine-tuning*). |
| **Autor** | _(completar)_ |

**Cómo ejecutar:** en Colab, `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`, y después `Entorno de ejecución → Ejecutar todas`. Duración aproximada con T4: 5–10 minutos (incluye descargas).

## 1. Configuración del entorno
Comprobamos la versión de Python, instalamos las dependencias que Colab no trae y registramos las versiones. **No se reinstala `torch`**: se usa la versión con CUDA que ya incluye Colab.

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Plataforma:", platform.platform())
IN_COLAB = "google.colab" in sys.modules
print("¿Ejecutando en Google Colab?:", IN_COLAB)

In [ ]:
# Dependencias. transformers y datasets se fijan para reproducibilidad; sentencepiece
# es necesario para el tokenizador de XLM-RoBERTa.
%pip install -q "transformers==5.19.0" "datasets==5.1.0" "sentencepiece==0.2.2" "scikit-learn" "psutil"

In [ ]:
import gc, json, os, random, time, datetime as dt
import importlib.metadata as md
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

VERSIONS = {"python": platform.python_version()}
for pkg in ["torch", "transformers", "datasets", "tokenizers", "sentencepiece",
            "scikit-learn", "pandas", "numpy", "matplotlib", "psutil", "huggingface_hub"]:
    try:
        VERSIONS[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        VERSIONS[pkg] = "no instalado"
pd.Series(VERSIONS, name="versión").to_frame()

In [ ]:
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

RESULTS_DIR = Path("results/colab_gpu" if IN_COLAB else "results/notebook_local")
(RESULTS_DIR / "figures").mkdir(parents=True, exist_ok=True)
print("Semilla:", SEED, "| resultados en:", RESULTS_DIR.resolve())

## 2. Configuración de GPU
Se verifica CUDA y se identifica la GPU. Si no hay GPU, el notebook **sigue funcionando en CPU** (o MPS en Mac), pero lo informa explícitamente: los tiempos obtenidos no deben presentarse como tiempos de GPU.

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    DEVICE_DESC = torch.cuda.get_device_name(0)
    print(f"GPU: {DEVICE_DESC} | memoria total: {props.total_memory/1024**3:.1f} GB | CUDA (build de torch): {torch.version.cuda}")
    !nvidia-smi --query-gpu=name,driver_version,memory.total,memory.used --format=csv
elif torch.backends.mps.is_available():
    DEVICE, DEVICE_DESC = torch.device("mps"), "Apple MPS"
    print("⚠️ Sin CUDA: se usará MPS (Apple). No equivale a la GPU de Colab.")
else:
    DEVICE, DEVICE_DESC = torch.device("cpu"), f"CPU ({platform.processor() or platform.machine()})"
    print("⚠️ GPU NO disponible: se ejecutará en CPU. En Colab: Entorno de ejecución → Cambiar tipo → GPU.")
print("Dispositivo seleccionado:", DEVICE, "-", DEVICE_DESC)
print(f"RAM del sistema: {psutil.virtual_memory().total/1024**3:.1f} GB | CPUs: {psutil.cpu_count()}")

def synchronize():
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    elif DEVICE.type == "mps": torch.mps.synchronize()

## 3. Autenticación en Hugging Face Hub
Los tres modelos y el dataset son **públicos y no restringidos (*gated*)**, por lo que **no se requiere autenticación** para descargarlos. Sin un token, el Hub puede aplicar límites de descarga más estrictos y mostrar un aviso, sin afectar los resultados.

Si se desea autenticar (por ejemplo, para recursos privados o *gated*), la forma segura en Colab es:
1. Crear un token de **solo lectura** en <https://huggingface.co/settings/tokens>.
2. Guardarlo en el panel **🔑 Secretos** de Colab con el nombre `HF_TOKEN` y habilitar el acceso del notebook.
3. Nunca escribir el token en una celda, ni imprimirlo, ni subirlo a GitHub.

La celda siguiente usa el secreto solo si existe; en caso contrario continúa de forma anónima.

In [ ]:
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Autenticado con el secreto HF_TOKEN (valor no mostrado).")
else:
    print("Sin token: acceso anónimo a recursos públicos.")

# Verificación de acceso a los recursos (metadatos públicos del Hub)
from huggingface_hub import HfApi
api = HfApi()
DATASET_ID = "SetFit/amazon_reviews_multi_es"
DATASET_REVISION = "16015418b488c9186fce74b058877ea939ca934d"
MODELS = {
    "nlptown/bert-base-multilingual-uncased-sentiment": "8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d",
    "cardiffnlp/twitter-xlm-roberta-base-sentiment": "f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8",
    "lxyuan/distilbert-base-multilingual-cased-sentiments-student": "cf991100d706c13c0a080c097134c05b7f436c45",
}
SHORT = {m: n for m, n in zip(MODELS, ["mBERT-nlptown", "XLM-R-cardiffnlp", "DistilmBERT-lxyuan"])}
print("Dataset:", api.dataset_info(DATASET_ID, revision=DATASET_REVISION).id, "| privado:", api.dataset_info(DATASET_ID).private)
for m, rev in MODELS.items():
    info = api.model_info(m, revision=rev)
    print(f"Modelo: {m} | gated: {info.gated} | privado: {info.private} | tarea: {info.pipeline_tag}")

## 4. Carga y preparación del dataset
- **Fuente:** `SetFit/amazon_reviews_multi_es`, commit fijado `1601541`. Es una réplica de la partición en español del *Multilingual Amazon Reviews Corpus* (Keung et al., 2020); el dataset original `amazon_reviews_multi` ya no se distribuye desde el Hub.
- **Partición:** se usa la partición **`test` oficial** completa (5 000 reseñas). Como no se entrena ningún modelo, no hay riesgo de contaminación entrenamiento→prueba dentro de este experimento (ver riesgos sobre los datos de entrenamiento de los modelos).
- **Transformación de etiquetas:** `label` (0–4) → estrellas (1–5) → sentimiento: 1–2 = negativo, 3 = neutral, 4–5 = positivo.
- **Licencia:** uso académico; prohíbe republicar el corpus. Por ello, el CSV con textos se genera localmente y no se publica en GitHub.

In [ ]:
ds = load_dataset(DATASET_ID, split="test", revision=DATASET_REVISION)
print(ds)
df = ds.to_pandas()
df.head()

In [ ]:
# Validación de campos, faltantes y duplicados
assert {"id", "text", "label", "label_text"} <= set(df.columns), "Faltan columnas"
print("Tipos:\n", df.dtypes, sep="")
print("\nValores faltantes por columna:\n", df.isna().sum(), sep="")
print("Textos vacíos:", int((df["text"].str.strip() == "").sum()))
print("IDs duplicados:", int(df["id"].duplicated().sum()))
dups = df[df["text"].duplicated(keep=False)].sort_values("text")
print("Filas con texto duplicado:", int(df["text"].duplicated().sum()),
      "| textos duplicados con etiquetas distintas:", int(dups.groupby("text")["label"].nunique().gt(1).sum()))

In [ ]:
# Transformación y verificación de etiquetas
LABELS = ["negativo", "neutral", "positivo"]
def stars_to_label(s):
    return "negativo" if s <= 2 else ("neutral" if s == 3 else "positivo")

assert (df["label_text"].astype(int) == df["label"]).all()
df["stars"] = df["label"].astype(int) + 1
assert df["stars"].between(1, 5).all()
df["sentiment"] = df["stars"].map(stars_to_label)
df = df.rename(columns={"id": "review_id"})
df["n_words"] = df["text"].str.split().str.len()

dist = pd.concat({"estrellas": df["stars"].value_counts().sort_index()}, axis=1)
print(dist.T)
print("\nDistribución de sentimiento (clases objetivo):")
print(df["sentiment"].value_counts().reindex(LABELS).to_frame("n").assign(pct=lambda x: (x.n / x.n.sum() * 100).round(1)))
print("\nPalabras por reseña:", df["n_words"].describe().round(1).to_dict())

In [ ]:
# Archivo de datos utilizado en la evaluación (solo uso local; no se publica)
Path("data/processed").mkdir(parents=True, exist_ok=True)
df["sentiment_id"] = df["sentiment"].map({l: i for i, l in enumerate(LABELS)})
df["n_chars"] = df["text"].str.len()
# Mismas columnas que src/prepare_data.py
df[["review_id", "text", "stars", "sentiment", "sentiment_id", "n_chars", "n_words"]].to_csv(
    "data/processed/dataset.csv", index=False)
df[["review_id", "stars", "sentiment"]].to_csv("data/processed/dataset_manifest.csv", index=False)
print("Registros para evaluación:", len(df))

**Conclusión parcial (datos).** El conjunto de prueba está balanceado en estrellas (1 000 por nivel), pero al agrupar en tres clases queda 40 % negativo / 20 % neutral / 40 % positivo. Este desbalance moderado hace que *accuracy* sea insuficiente por sí sola: se usará **F1 macro** como métrica principal, porque pondera por igual la clase neutral, que es la más difícil y la menos frecuente. No hay faltantes; los pocos duplicados se conservan para no alterar la partición oficial.

## 5. Selección y carga de modelos
| # | Identificador (commit fijado) | Arquitectura | Salida original | Normalización a 3 clases |
|---|---|---|---|---|
| 1 | `nlptown/bert-base-multilingual-uncased-sentiment` (`8f6f4e3`) | BERT-base multilingüe, *uncased* | 1–5 estrellas | 1–2 → negativo · 3 → neutral · 4–5 → positivo (misma regla que el dataset) |
| 2 | `cardiffnlp/twitter-xlm-roberta-base-sentiment` (`f2f1202`) | XLM-RoBERTa-base (XLM-T) | negative/neutral/positive | directa |
| 3 | `lxyuan/distilbert-base-multilingual-cased-sentiments-student` (`cf99110`) | DistilBERT multilingüe (destilado) | positive/neutral/negative | directa (ojo: orden de índices distinto) |

La normalización se hace **por nombre de etiqueta** (`config.id2label`), nunca por índice, porque cada modelo ordena sus clases de forma distinta.

In [ ]:
MODEL_LABEL_MAP = {"1 star": "negativo", "2 stars": "negativo", "3 stars": "neutral",
                   "4 stars": "positivo", "5 stars": "positivo",
                   "negative": "negativo", "neutral": "neutral", "positive": "positivo"}

def load_model(model_id, revision):
    try:
        tok = AutoTokenizer.from_pretrained(model_id, revision=revision)
        mdl = AutoModelForSequenceClassification.from_pretrained(model_id, revision=revision)
    except Exception as exc:
        raise RuntimeError(f"No se pudo cargar {model_id}@{revision}: {exc}") from exc
    mdl.to(DEVICE).eval()
    id2label = {int(k): v for k, v in mdl.config.id2label.items()}
    unknown = [v for v in id2label.values() if v.lower() not in MODEL_LABEL_MAP]
    if unknown:
        raise ValueError(f"Etiquetas sin mapeo en {model_id}: {unknown}")
    return tok, mdl, {k: MODEL_LABEL_MAP[v.lower()] for k, v in id2label.items()}

# Inspección (sin cronometrar): clases de salida, parámetros y tamaño en el Hub
model_info_rows = []
for m, rev in MODELS.items():
    tok, mdl, id2target = load_model(m, rev)
    files = {s.rfilename: s.size for s in api.model_info(m, revision=rev, files_metadata=True).siblings}
    w = files.get("model.safetensors") or files.get("pytorch_model.bin")
    model_info_rows.append({"modelo": SHORT[m], "model_id": m, "revision": rev[:7],
                            "tipo": mdl.config.model_type, "tokenizador": type(tok).__name__,
                            "id2label": mdl.config.id2label, "parametros_M": round(sum(p.numel() for p in mdl.parameters())/1e6, 1),
                            "pesos_hub_MB": round(w/1024**2, 1) if w else None})
    del tok, mdl; gc.collect()
pd.DataFrame(model_info_rows)

## 6. Pipeline de inferencia y medición de latencia
Condiciones **idénticas** para los tres modelos: mismos 5 000 registros en el mismo orden, `batch_size=32`, `max_length=256` tokens (trunca < 0,3 % de las reseñas), precisión FP32, `torch.inference_mode()`, *padding* dinámico por lote.

Mediciones por modelo:
1. **Tiempo de carga** (descarga/lectura + traslado al dispositivo), separado del tiempo de inferencia.
2. **Calentamiento:** 3 lotes que no se cronometran.
3. **Inferencia en lote:** 3 pasadas completas cronometradas (se reporta media y desviación estándar), con sincronización del dispositivo antes de detener el reloj.
4. **Inferencia en línea:** 200 reseñas procesadas una a una (`batch_size=1`) para estimar la latencia percibida por un usuario (media, p50, p95).
5. **Memoria pico** de la GPU (`torch.cuda.max_memory_allocated`).

In [ ]:
BATCH_SIZE, MAX_LENGTH, WARMUP_BATCHES, REPEATS, ONLINE_N = 32, 256, 3, 3, 200
texts = df["text"].tolist()
y_true = df["sentiment"].tolist()
online_texts = df.sample(n=ONLINE_N, random_state=SEED)["text"].tolist()

@torch.inference_mode()
def predict(tok, mdl, batch_texts, batch_size=BATCH_SIZE):
    out = []
    for i in range(0, len(batch_texts), batch_size):
        enc = tok(batch_texts[i:i+batch_size], padding=True, truncation=True,
                  max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        out.append(mdl(**enc).logits.argmax(-1).cpu())
    synchronize()
    return torch.cat(out).numpy()

def mem_mb():
    if DEVICE.type == "cuda": return torch.cuda.max_memory_allocated() / 1024**2
    if DEVICE.type == "mps": return torch.mps.driver_allocated_memory() / 1024**2
    return psutil.Process().memory_info().rss / 1024**2

def run_model(model_id, revision):
    name = SHORT[model_id]
    print(f"\n=== {name} ({model_id})")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    m0 = mem_mb()
    t0 = time.perf_counter()
    tok, mdl, id2target = load_model(model_id, revision)
    synchronize(); load_s = time.perf_counter() - t0

    predict(tok, mdl, texts[:BATCH_SIZE * WARMUP_BATCHES])       # calentamiento
    times, raw = [], None
    for r in range(REPEATS):
        t0 = time.perf_counter()
        out = predict(tok, mdl, texts)
        times.append(time.perf_counter() - t0)
        raw = out if raw is None else raw
        print(f"  pasada {r+1}: {times[-1]:.2f} s")
    peak = mem_mb()
    online = []
    for t in online_texts:
        t0 = time.perf_counter(); predict(tok, mdl, [t], batch_size=1)
        online.append((time.perf_counter() - t0) * 1000)

    y_pred = [id2target[i] for i in raw]
    acc = accuracy_score(y_true, y_pred)
    pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average="weighted", zero_division=0)
    rep = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)
    total = float(np.mean(times))
    row = {"modelo": name, "model_id": model_id, "revision": revision,
           "accuracy": acc, "precision_macro": pm, "recall_macro": rm, "f1_macro": fm,
           "precision_weighted": pw, "recall_weighted": rw, "f1_weighted": fw,
           "recall_negativo": rep["negativo"]["recall"], "f1_neutral": rep["neutral"]["f1-score"],
           "n_registros": len(texts), "tiempo_total_s": total, "tiempo_total_std_s": float(np.std(times)),
           "latencia_media_ms": total / len(texts) * 1000, "rendimiento_reg_s": len(texts) / total,
           "online_media_ms": float(np.mean(online)), "online_p50_ms": float(np.percentile(online, 50)),
           "online_p95_ms": float(np.percentile(online, 95)), "tiempo_carga_s": load_s,
           "parametros_M": sum(p.numel() for p in mdl.parameters()) / 1e6,
           "memoria_pico_MB": peak, "memoria_incremento_MB": peak - m0,
           "dispositivo": DEVICE_DESC, "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH}
    if model_id.startswith("nlptown/"):
        stars_pred = raw + 1
        row["stars_exact_acc"] = float((stars_pred == df["stars"].values).mean())
        row["stars_off_by_1_acc"] = float((np.abs(stars_pred - df["stars"].values) <= 1).mean())
    del tok, mdl; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return row, rep, y_pred

### 6.1 Modelo 1 — `nlptown/bert-base-multilingual-uncased-sentiment`

In [ ]:
results, reports, preds = {}, {}, {}
m = "nlptown/bert-base-multilingual-uncased-sentiment"
results[m], reports[m], preds[m] = run_model(m, MODELS[m])

### 6.2 Modelo 2 — `cardiffnlp/twitter-xlm-roberta-base-sentiment`

In [ ]:
m = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
results[m], reports[m], preds[m] = run_model(m, MODELS[m])

### 6.3 Modelo 3 — `lxyuan/distilbert-base-multilingual-cased-sentiments-student`

In [ ]:
m = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
results[m], reports[m], preds[m] = run_model(m, MODELS[m])

## 7. Evaluación del desempeño
- **Accuracy**: proporción de aciertos.
- **Precision, recall y F1** por clase y promediados:
  - **macro** (promedio simple de las 3 clases): **criterio principal de selección**, porque el caso de uso necesita reconocer bien las tres clases y la neutral es minoritaria (20 %).
  - **weighted** (ponderado por soporte): se reporta como referencia; se parece a *accuracy* y favorece a las clases mayoritarias.
- **Recall de la clase negativa**: métrica de negocio, ya que una reseña negativa no detectada no se escala a servicio al cliente.

Todas las métricas se calculan con `scikit-learn`.

In [ ]:
comp = pd.DataFrame(results.values())
metric_cols = ["modelo", "accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted", "recall_negativo", "f1_neutral"]
comp[metric_cols].round(4)

In [ ]:
# Reporte por clase
per_class = pd.DataFrame([{"modelo": SHORT[m], "clase": c, "precision": r[c]["precision"], "recall": r[c]["recall"],
                           "f1": r[c]["f1-score"], "soporte": int(r[c]["support"])}
                          for m, r in reports.items() for c in LABELS])
per_class.pivot(index="clase", columns="modelo", values=["precision", "recall", "f1"]).round(3)

In [ ]:
# Matrices de confusión (conteos y % por fila)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, (m, yp) in zip(axes, preds.items()):
    cm = confusion_matrix(y_true, yp, labels=LABELS)
    norm = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i,j]}\n({norm[i,j]:.0%})", ha="center", va="center",
                    color="white" if norm[i,j] > 0.6 else "black", fontsize=9)
    ax.set_xticks(range(3), LABELS); ax.set_yticks(range(3), LABELS)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
    ax.set_title(f"{SHORT[m]}\nF1 macro = {results[m]['f1_macro']:.3f}", fontsize=10)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "figures/confusion_matrices.png", dpi=150); plt.show()

In [ ]:
# Clase con peor desempeño por modelo y ejemplos de errores (texto truncado a 100 caracteres
# para no republicar el corpus; ver condiciones de uso del dataset)
for m in MODELS:
    pc = per_class[per_class.modelo == SHORT[m]].set_index("clase")
    print(f"{SHORT[m]}: clase con menor F1 = {pc['f1'].idxmin()} ({pc['f1'].min():.3f})")

pred_df = df[["review_id", "text", "stars", "sentiment"]].copy()
for m in MODELS:
    pred_df[f"pred_{SHORT[m]}"] = preds[m]
pred_cols = [f"pred_{SHORT[m]}" for m in MODELS]
all_wrong = pred_df[pred_df[pred_cols].ne(pred_df["sentiment"], axis=0).all(axis=1)]
print(f"\nReseñas mal clasificadas por los tres modelos: {len(all_wrong)} ({len(all_wrong)/len(df):.1%})")
print(all_wrong["stars"].value_counts().sort_index().rename("por estrellas").to_string())
all_wrong.assign(text=all_wrong["text"].str.slice(0, 100)).sample(5, random_state=SEED)

**Conclusión parcial (calidad).** _Interpretar aquí con los valores obtenidos en esta ejecución:_ comparar F1 macro, revisar si la clase neutral concentra los errores (esperable: una reseña de 3 estrellas suele mezclar aspectos positivos y negativos) y observar si los errores comunes a los tres modelos corresponden a reseñas de 2 o 4 estrellas (frontera difusa de la regla de agrupación).

## 8. Latencia y recursos

In [ ]:
lat_cols = ["modelo", "tiempo_carga_s", "tiempo_total_s", "tiempo_total_std_s", "latencia_media_ms",
            "rendimiento_reg_s", "online_media_ms", "online_p50_ms", "online_p95_ms",
            "parametros_M", "memoria_pico_MB", "dispositivo", "batch_size"]
comp[lat_cols].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for _, r in comp.iterrows():
    ax.scatter(r["rendimiento_reg_s"], r["f1_macro"], s=r["parametros_M"] * 3, alpha=0.7)
    ax.annotate(f"{r['modelo']}\n{r['parametros_M']:.0f} M parám.", (r["rendimiento_reg_s"], r["f1_macro"]),
                textcoords="offset points", xytext=(8, 6), fontsize=8)
ax.axhline(0.60, ls="--", lw=1, color="gray")
ax.set_xlabel("Rendimiento (reseñas/s, lote de 32)"); ax.set_ylabel("F1 macro")
ax.set_title(f"Calidad vs. eficiencia — {DEVICE_DESC}", fontsize=10); ax.margins(x=0.35, y=0.25)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "figures/f1_vs_throughput.png", dpi=150); plt.show()

**Conclusión parcial (eficiencia).** _Interpretar aquí:_ el número de parámetros (DistilBERT ≈ 135 M < BERT ≈ 167 M < XLM-R ≈ 278 M) anticipa el costo, pero la latencia real también depende del tokenizador y la longitud media de secuencia. El costo monetario en Colab gratuito es 0, pero la disponibilidad de GPU no está garantizada; se documentan los recursos observados, no costos.

## 9. Comparación de resultados y verificación de umbrales
Umbrales **propuestos** antes de la ejecución (no validados por una organización real):

| Criterio | Umbral | Justificación |
|---|---|---|
| F1 macro | ≥ 0.60 | Nivel mínimo útil para triaje en 3 clases con clase neutral ambigua. |
| Recall negativo | ≥ 0.75 | Al menos 3 de cada 4 quejas deben escalarse. |
| Rendimiento en lote (GPU) | ≥ 50 reseñas/s | Procesar 100 000 reseñas diarias en < 35 min. |
| Latencia p95 en línea (GPU) | ≤ 100 ms | Clasificación percibida como inmediata en un panel. |
| Memoria pico | ≤ 2 GB | Cabe junto con otros servicios en una GPU de 16 GB (T4). |

In [ ]:
TH = {"f1_macro": (">=", 0.60), "recall_negativo": (">=", 0.75), "rendimiento_reg_s": (">=", 50.0),
      "online_p95_ms": ("<=", 100.0), "memoria_pico_MB": ("<=", 2048.0)}
table = comp[["modelo", "accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted",
              "latencia_media_ms", "rendimiento_reg_s", "online_p95_ms", "parametros_M", "memoria_pico_MB", "dispositivo"]].copy()
for k, (op, v) in TH.items():
    table[f"cumple_{k}"] = comp[k] >= v if op == ">=" else comp[k] <= v
if DEVICE.type != "cuda":
    print("⚠️ Los umbrales de latencia/rendimiento están definidos para GPU CUDA; esta ejecución no usó CUDA.")
table.round(4)

In [ ]:
# Exportar resultados de esta ejecución
comp.round(4).to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
per_class.round(4).to_csv(RESULTS_DIR / "per_class_metrics.csv", index=False)
pred_df.drop(columns="text").to_csv(RESULTS_DIR / "predictions.csv", index=False)
meta = {"generated_at": dt.datetime.now().isoformat(timespec="seconds"), "device": str(DEVICE),
        "device_description": DEVICE_DESC, "versions": VERSIONS, "seed": SEED,
        "dataset": {"id": DATASET_ID, "revision": DATASET_REVISION, "split": "test", "n_records": len(df)},
        "models": MODELS, "inference": {"batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
        "warmup_batches": WARMUP_BATCHES, "timed_repeats": REPEATS, "online_samples": ONLINE_N}}
(RESULTS_DIR / "run_metadata.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print("Archivos:", sorted(str(p) for p in RESULTS_DIR.rglob("*") if p.is_file()))
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("colab_gpu_results", "zip", RESULTS_DIR)
    files.download("colab_gpu_results.zip")  # copiar su contenido a results/colab_gpu/ del repositorio

## 10. Conclusión final
_Esta sección se redacta a partir de las tablas anteriores; los valores de referencia y el análisis completo están en `docs/analisis_comparativo.docx` y en `results/`._ Con los resultados de la ejecución de referencia (ver README):

- **Mejor desempeño predictivo:** el modelo con mayor F1 macro de la sección 9.
- **Mejor eficiencia:** el de mayor rendimiento y menor latencia de la sección 8.
- **Mejor equilibrio:** el que cumple todos los umbrales con mayor F1 macro.
- **Recomendado:** ver la justificación en el documento Word (debe coincidir con las tablas de esta ejecución; si no coincide, documentar la diferencia).

**Limitaciones:** solo inferencia (sin *fine-tuning*); reseñas de amazon.es (español de España) y no de la organización destino; la regla 3 estrellas = neutral es una convención; la clase neutral es inherentemente ambigua; los tiempos dependen de la GPU asignada por Colab.

**Riesgos:** sesgo por dominio/variante dialectal, deriva de datos, licencias (el modelo XLM-R de Cardiff no declara licencia en su *model card*; el corpus prohíbe uso comercial), y privacidad de las reseñas reales.

**Mejoras:** *fine-tuning* del mejor candidato con reseñas propias etiquetadas, calibración de umbrales por clase, cuantización/ONNX para CPU y una demo en Hugging Face Spaces (Gradio).

## 11. Reproducibilidad
| Elemento | Valor |
|---|---|
| Semilla | 42 (Python, NumPy, PyTorch) |
| Dataset | `SetFit/amazon_reviews_multi_es` @ `16015418b488c9186fce74b058877ea939ca934d`, partición `test` (5 000) |
| Modelo 1 | `nlptown/bert-base-multilingual-uncased-sentiment` @ `8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d` |
| Modelo 2 | `cardiffnlp/twitter-xlm-roberta-base-sentiment` @ `f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8` |
| Modelo 3 | `lxyuan/distilbert-base-multilingual-cased-sentiments-student` @ `cf991100d706c13c0a080c097134c05b7f436c45` |
| Inferencia | FP32, `batch_size=32`, `max_length=256`, 3 lotes de calentamiento, 3 repeticiones, 200 muestras en línea |
| Versiones | tabla de la sección 1 y `run_metadata.json` |

**Para reproducir desde cero:** `Entorno de ejecución → Desconectar y borrar el entorno de ejecución`, volver a conectar con GPU y `Ejecutar todas`. Fuera de Colab: ver `README.md` (scripts `src/prepare_data.py` y `src/evaluate_models.py`, que implementan el mismo procedimiento).

La inferencia en FP32 es determinista en las predicciones salvo diferencias numéricas mínimas entre tipos de GPU; los tiempos varían según el hardware asignado.

In [ ]:
print(json.dumps({"versions": VERSIONS, "device": DEVICE_DESC}, indent=2, ensure_ascii=False))